# Experimentos de OpenVLA sobre LIBERO

Corre OpenVLA en varias tareas de LIBERO y muestra los resultados (tabla, tasa
de éxito y videos). El código pesado vive en el framework (`core`, `benchmarks`,
`models`); este notebook solo hace **configuración + ejecución + visualización**.

> Requiere un entorno con **GPU** y OpenVLA + LIBERO instalados (jupyter en el
> cluster, o Kaggle). Los pesos deben poder descargarse/estar en `HF_HOME`.

In [ ]:
!pip install "pandas==2.2.3" "matplotlib-inline==0.1.7" "numpy<2"

## 1. Setup

In [2]:
import os, sys, glob

os.environ.setdefault("MUJOCO_GL", "egl")     # render headless (sin pantalla)

# --- Shim de torch.load (necesario con GPUs nuevas) ---
# Con GPU nueva (Blackwell) se necesita torch >= 2.6, cuyo torch.load usa por
# defecto weights_only=True; eso rompe la carga de los checkpoints de
# OpenVLA/LIBERO (pickles con numpy) -> "UnpicklingError: Weights only load
# failed". Restauramos el comportamiento anterior. Seguro aqui: los checkpoints
# de OpenVLA y LIBERO son de fuente confiable. Debe ir ANTES de cargar modelo
# o entorno.
import functools
import torch
torch.load = functools.partial(torch.load, weights_only=False)


def _find_project_root():
    """Sube desde el cwd buscando la raiz del proyecto (simulation.py + core/)."""
    d = os.getcwd()
    for _ in range(6):
        if os.path.exists(os.path.join(d, "simulation.py")) and \
           os.path.isdir(os.path.join(d, "core")):
            return d
        d = os.path.dirname(d)
    hits = glob.glob(os.path.join(os.getcwd(), "**", "simulation.py"), recursive=True)
    if hits:
        return os.path.dirname(os.path.abspath(hits[0]))
    raise RuntimeError("No encontre la raiz del proyecto (simulation.py + core/).")


ROOT = _find_project_root()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
os.environ.setdefault("HF_HOME", os.path.join(ROOT, "hf_cache"))   # caché de pesos
print("Proyecto:", ROOT)
print("HF_HOME :", os.environ["HF_HOME"])

Proyecto: /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation
HF_HOME : /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/hf_cache


## 2. Configuración

In [3]:
from dataclasses import dataclass


@dataclass
class Config:
    suite: str = "libero_10"
    num_tasks: int = 10             # cuantos escenarios (tareas) probar
    episodes_per_task: int = 2     # configuraciones iniciales por tarea
    max_steps: int = 520           # libero_10 es de horizonte largo (oficial ~520)
    out_dir: str = "output/experiments"


cfg = Config()
cfg

Config(suite='libero_10', num_tasks=10, episodes_per_task=2, max_steps=520, out_dir='output/experiments')

## 3. Cargar el modelo (una sola vez)

In [4]:
from core import View
from models.OpenVLA import OpenVLAController

model = OpenVLAController(view=View.AGENT, center_crop=True, device="cuda")
print("modelo cargado | unnorm_key =", model.unnorm_key)

Empiezo load


/mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
The model weights are not tied. Please use the `tie_weights` method before using the `infer_auto_device` function.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


modelo cargado | unnorm_key = libero_10


## 4. Correr los experimentos
Un `LiberoController` por tarea (escenario); `run_experiments` corre los
episodios y graba un video por cada uno.

In [ ]:
from core import run_experiments
from benchmarks.libero import LiberoController

task_list = LiberoController.tasks(cfg.suite)[:cfg.num_tasks]
benchmarks = [LiberoController(task_id=tid, suite=cfg.suite) for tid, _ in task_list]

results = run_experiments(
    model, benchmarks,
    max_steps=cfg.max_steps,
    episodes_per_task=cfg.episodes_per_task,
    out_dir=cfg.out_dir,
    record_view=View.AGENT,
)
print(f"{len(results)} episodios corridos")

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /mnt-homes/wapol/asoto/diegoftpxd/miniforge3/envs/openvla/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r/libero/libero/../datasets does not exist!
[Warning]: datasets path /mnt-homes/wapol/asoto/diegoftpxd/MuJoCo-simulation/benchmarks/libero/Libero-10-r

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


video guardado: output/experiments/scenario0_ep0.mp4 (312 cuadros)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


...

video guardado: output/experiments/scenario3_ep0.mp4 (520 cuadros)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


video guardado: output/experiments/scenario3_ep1.mp4 (520 cuadros)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


video guardado: output/experiments/scenario4_ep0.mp4 (239 cuadros)


## 5. Resultados — tabla y tasa de éxito

In [ ]:
import pandas as pd
from core import summarize

df = pd.DataFrame(results)
resumen = summarize(results)
print("Tasa de exito global: {exitos}/{total} = {tasa_exito:.0%}".format(**resumen))
df[["scenario", "instruction", "episode", "success", "steps"]]

In [ ]:
import matplotlib.pyplot as plt

por_escenario = df.groupby("scenario")["success"].mean()
ax = por_escenario.plot(kind="bar", ylim=(0, 1), color="#4C78A8", rot=0)
ax.set_xlabel("escenario"); ax.set_ylabel("tasa de exito")
ax.set_title("Exito por tarea"); plt.tight_layout(); plt.show()

## 6. Resultados — videos

In [ ]:
from IPython.display import Video, display

for r in results:
    if r["video"]:
        print(f"escenario {r['scenario']} | {r['instruction']} | exito={r['success']}")
        display(Video(r["video"], embed=True, width=320))